# 第 19 章互动实验：迷你 PatchCore

先用简单的局部纹理特征把你实现的零件串成完整流程，不需要下载任何权重；中间两个“对照”单元把数据换难一点，让局部聚合的边界问题和核心集的代价显出来；最后一节可选地换成预训练 ResNet 特征，拼成完整的 PatchCore。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "19_异常检测与深度缺陷检测"
ASSIGNMENT = "19-1_PatchCore核心.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def make_image(seed, defect=False, aligned=True, amp=0.2):
    """一张 64x64 的条纹图。aligned=True 时条纹相位只在很小的范围内变化（相当于已经对齐好），False 时相位完全随机；amp 是缺陷的灰度幅度。"""
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:64, 0:64]
    phase = rng.uniform(0, 0.3) if aligned else rng.uniform(0, 2 * np.pi)
    img = 0.5 + 0.3 * np.sin(2 * np.pi * x / 8 + phase) + rng.normal(scale=0.06, size=(64, 64))
    mask = np.zeros((64, 64), bool)
    if defect:
        cy, cx = rng.integers(10, 54, 2); r = rng.integers(2, 4)
        mask = (y - cy) ** 2 + (x - cx) ** 2 <= r * r
        img[mask] += rng.choice([-1.0, 1.0]) * amp
    return img, mask

def patch_features(img, p=4, agg="edge"):
    """把每个 4x4 patch（步长 2）展平成 16 维向量，再做 3x3 邻域平均（模仿 PatchCore 的局部聚合）。

    agg="edge"：图像边界上的 patch 邻域不完整，用复制边缘的办法补齐（默认）；
    agg="valid"：只对真实存在的邻居平均；agg=None：不做聚合。后两种只在“对照 1”里用到。
    """
    pats = np.lib.stride_tricks.sliding_window_view(img, (p, p))[::2, ::2]
    feats = pats.reshape(pats.shape[0], pats.shape[1], -1).astype(np.float64)
    if agg is None:
        return feats
    gh, gw = feats.shape[:2]
    if agg == "valid":
        pad, ones = np.pad(feats, ((1, 1), (1, 1), (0, 0))), np.pad(np.ones((gh, gw)), 1)
        count = sum(ones[dy:dy + gh, dx:dx + gw] for dy in range(3) for dx in range(3))
        return sum(pad[dy:dy + gh, dx:dx + gw] for dy in range(3) for dx in range(3)) / count[..., None]
    pad = np.pad(feats, ((1, 1), (1, 1), (0, 0)), mode="edge")
    return sum(pad[dy:dy + gh, dx:dx + gw] for dy in range(3) for dx in range(3)) / 9

def score_maps(memory, images, agg="edge"):
    """对每张图算 31x31 的 patch 分数图，返回 (n, 31, 31)。"""
    out = []
    for im in images:
        f = patch_features(im, agg=agg)
        out.append(mod.knn_scores(memory, f.reshape(-1, 16), k=1).reshape(f.shape[:2]))
    return np.stack(out)

train = [make_image(s)[0] for s in range(20)]
bank = np.concatenate([patch_features(im).reshape(-1, 16) for im in train])
print("记忆库大小：", bank.shape)

In [ ]:
idx = attempt(mod.greedy_coreset, bank, max(1, len(bank) // 20))
img_scores = None
if idx is not None:
    try:
        memory = bank[idx]
        print("核心集大小：", memory.shape)
        test = [make_image(1000 + s, defect=(s % 2 == 0)) for s in range(40)]
        labels = np.array([int(m.any()) for _, m in test])
        maps = score_maps(memory, [im for im, _ in test])
        img_scores = maps.reshape(len(maps), -1).max(axis=1)
        print("Image AUROC：", round(mod.auroc(img_scores, labels), 4))
        fig, ax = plt.subplots(2, 4, figsize=(12, 6))
        for i in range(4):
            ax[0, i].imshow(test[i][0], cmap="gray"); ax[0, i].contour(test[i][1], colors="c")
            ax[1, i].imshow(maps[i], cmap="magma"); ax[1, i].set_title(f"分数 {img_scores[i]:.2f}")
        plt.show()
    except NotImplementedError as exc:
        img_scores = None
        print("先完成作业里剩下的函数：", exc)

## 对照 1：良品的最高分落在哪（局部聚合的边界问题）

上面第二行的分数图里，没有缺陷的图最亮的地方总在最外一圈。原因在 `patch_features`：图像边界上的 patch 没有完整的 3×3 邻域，默认用复制边缘的办法补齐，补出来的邻居就是它自己，平均时噪声压得没有内部那么低，分数就偏高。这是讲义第 3.2 节的内容，所以 Notebook 保留了这种补法，没有把它“修掉”。

上面那组数据是对齐的，缺陷也明显，这个毛病不影响 AUROC。下面的单元先数一下“良品图的最高分有多少落在最外一圈”，再换成难一点的数据（条纹相位完全随机、缺陷幅度 0.12），比较三种聚合方式和两种取整图分数的办法。

先预测：换成“只对真实存在的邻居平均”，最外一圈的问题会消失吗？去掉最外一圈之后，聚合和不聚合哪个 AUROC 高？

In [ ]:
RING = np.ones((31, 31), bool); RING[1:-1, 1:-1] = False      # 31x31 分数图的最外一圈

def ring_report(score_stack, y):
    """返回：良品图的最高分落在最外一圈的比例、全图取最大的 image AUROC、去掉最外一圈再取最大的 image AUROC。"""
    flat = score_stack.reshape(len(score_stack), -1)
    on_ring = RING.ravel()[flat[y == 0].argmax(axis=1)].mean()
    return on_ring, mod.auroc(flat.max(axis=1), y), mod.auroc(flat[:, ~RING.ravel()].max(axis=1), y)

hard = None
if img_scores is not None:
    print(f"最外一圈占 patch 总数的 {RING.mean():.1%}")
    r = ring_report(maps, labels)
    print(f"上面那组数据（对齐、缺陷幅度 0.2）：良品最高分落在最外一圈的比例 {r[0]:.0%}，AUROC 全图取最大 {r[1]:.4f}，去掉最外一圈 {r[2]:.4f}")
    hard_train = [make_image(s, aligned=False, amp=0.12)[0] for s in range(20)]
    hard_test = [make_image(2000 + s, defect=(s % 2 == 0), aligned=False, amp=0.12) for s in range(100)]
    hard_labels = np.array([int(m.any()) for _, m in hard_test])
    hard = {}
    print("换成不对齐、缺陷幅度 0.12 的数据（20 张良品，100 张待检图，贪心核心集 10%）：")
    for agg, name in (("edge", "聚合，复制填充（默认）"), ("valid", "聚合，只对真实存在的邻居平均"), (None, "不聚合")):
        hb = np.concatenate([patch_features(im, agg=agg).reshape(-1, 16) for im in hard_train])
        order = mod.greedy_coreset(hb, len(hb) // 10)
        hm = score_maps(hb[order], [im for im, _ in hard_test], agg=agg)
        hard[agg] = (hb, order, hm)
        r = ring_report(hm, hard_labels)
        print(f"  {name}：良品最高分落在最外一圈的比例 {r[0]:.0%}，AUROC 全图取最大 {r[1]:.3f}，去掉最外一圈 {r[2]:.3f}")
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    for a, (agg, name) in zip(ax, (("edge", "复制填充"), ("valid", "只对真实邻居平均"), (None, "不聚合"))):
        im = a.imshow(hard[agg][2][hard_labels == 0].mean(axis=0), cmap="viridis"); a.set_title(name); a.axis("off")
        plt.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle("良品的平均分数图（31x31 个 patch，三栏的色标范围不同）"); plt.show()
    print("对照讲义第 3.2 节的表读这几行。100 张待检图时 AUROC 的最小变化是 0.0004，一两个百分点以内的差别不要当真。")

## 实验：从 AUROC 到产线工作点

要求漏检率不超过 0%、5%、10% 时，过检率分别是多少？再看核心集比例从 100% 压到 1%，AUROC 掉了多少，想想为什么掉得这么少。

In [ ]:
if img_scores is not None:
    for esc in (0.0, 0.05, 0.10):
        r = attempt(mod.threshold_at_escape, img_scores[labels == 0], img_scores[labels == 1], esc)
        if r: print(f"漏检上限 {esc:.0%}：阈值 {r[0]:.3f}，实际漏检 {r[1]:.0%}，过检 {r[2]:.0%}")
    for frac in (1.0, 0.2, 0.05, 0.01):
        m = max(1, int(len(bank) * frac))
        mem = bank[mod.greedy_coreset(bank, m)] if frac < 1 else bank
        sc = score_maps(mem, [im for im, _ in test]).reshape(len(test), -1).max(axis=1)
        print(f"核心集比例 {frac:>5.0%}：AUROC {mod.auroc(sc, labels):.4f}")
    print("这组数据是对齐的，缺陷也明显，压到 1% 几乎看不出差别。40 张待检图时 AUROC 的最小变化是 0.0025，上面几行的差别只是一两对样本。")
    print("不要据此认为“核心集可以随便压”，下一个单元换一组数据再压一次。")

## 对照 2：换一组难一点的数据再压核心集

还是“对照 1”里那组不对齐、缺陷幅度 0.12 的数据，整图分数去掉最外一圈。把核心集从 10% 压到 0.3%，同时和“随机抽同样多的点”比较。先预测：贪心选出来的核心集覆盖半径更小，它的 AUROC 一定比随机抽的高吗？（讲义第 3.3 节）

In [ ]:
if hard is not None:
    hb, order, _ = hard["edge"]
    rng = np.random.default_rng(0)
    inner = lambda s: s[:, 1:-1, 1:-1].reshape(len(s), -1).max(axis=1)
    hard_imgs = [im for im, _ in hard_test]
    print(f"不对齐、缺陷幅度 0.12，记忆库 {len(hb)} 个 patch，100 张待检图，整图分数去掉最外一圈")
    for frac in (0.10, 0.03, 0.01, 0.003):
        m = max(1, int(len(hb) * frac))
        greedy = mod.auroc(inner(score_maps(hb[order[:m]], hard_imgs)), hard_labels)      # 贪心的前 m 个点就是大小为 m 的核心集
        randoms = [mod.auroc(inner(score_maps(hb[rng.choice(len(hb), m, replace=False)], hard_imgs)), hard_labels) for _ in range(3)]
        print(f"  保留 {frac:>5.1%}（{m:>4} 个点）：贪心 AUROC {greedy:.3f}，随机抽 3 次 " + "、".join(f"{v:.3f}" for v in randoms))
    print("和上一个单元对比：同样压到 1%，这里的 AUROC 比保留 10% 时低了多少？贪心每一行都比随机抽的高吗？")
    print("随机抽的 3 次之间也有波动，只看方向，不看第三位小数。讲义第 3.3 节的表是同一种设置下 3 个种子、每个 200 张待检图的平均。")

## 选做：换成预训练 ResNet 特征，拼成完整的 PatchCore

前面的“特征”是 4×4 的像素块。真正的 PatchCore 用预训练 CNN 的中间层：下面的单元取 ResNet-18 的 layer2 和 layer3，各做 3×3 平均（局部聚合），把 layer3 上采样到 layer2 的尺寸后拼成 384 维，其余步骤全部调用你在作业里写的函数：记忆库 → `greedy_coreset` → `knn_scores` → 上采样成热图 → `auroc` 和 `threshold_at_escape`。

需要 torch、torchvision 和 ResNet-18 的 ImageNet 权重（约 45 MB）。本机没有权重时，这个单元打印说明后跳过，不会报错，也不会自己去联网；想下载就把单元第一行的 `ALLOW_DOWNLOAD` 改成 `True`。有显卡时自动用显卡，没有就用 CPU，这个规模在 CPU 上几秒钟。

在 MVTec AD 上跑时，建议直接用 Anomalib，把它的结果和这里的迷你版对照。

In [ ]:
ALLOW_DOWNLOAD = False      # 本机没有 ResNet-18 权重时，改成 True 允许 torchvision 联网下载一次

def patchcore_resnet(train_imgs, test_pairs, keep=0.10):
    try:
        import torch
        import torch.nn.functional as F
        import torchvision
    except ImportError as exc:
        print("跳过：没有装 torch 或 torchvision（" + str(exc) + "）。装好后重新运行这个单元。")
        return None
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    ckpt = Path(torch.hub.get_dir()) / "checkpoints" / weights.url.rsplit("/", 1)[-1]
    if not ckpt.exists() and not ALLOW_DOWNLOAD:
        print("跳过：本机没有 ResNet-18 的预训练权重，这一节没有运行。")
        print("  找的位置：", ckpt)
        print("  想运行：把本单元第一行的 ALLOW_DOWNLOAD 改成 True 再运行（需要联网，约 45 MB，只下载一次）。")
        return None
    try:
        net = torchvision.models.resnet18(weights=weights).eval()
    except Exception as exc:
        print("跳过：权重加载失败（" + type(exc).__name__ + "：" + str(exc) + "）。检查网络，或者把权重文件放到", ckpt)
        return None
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = net.to(device)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    @torch.no_grad()
    def extract(images):
        """(n, 64, 64) 的灰度图 → (n, 16, 16, 384) 的 patch 特征。"""
        x = torch.tensor(np.stack(images), dtype=torch.float32, device=device)[:, None]
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False).clamp(0, 1).repeat(1, 3, 1, 1)   # 放大到 128x128，复制成 3 通道
        h = net.maxpool(net.relu(net.bn1(net.conv1((x - mean) / std))))
        f2 = net.layer2(net.layer1(h))                                   # 16x16，128 维
        f3 = net.layer3(f2)                                              # 8x8，256 维
        f2, f3 = F.avg_pool2d(f2, 3, 1, 1), F.avg_pool2d(f3, 3, 1, 1)    # 局部聚合：3x3 邻域平均
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f2, f3], dim=1).permute(0, 2, 3, 1).cpu().numpy().astype(np.float64)

    y = np.array([int(m.any()) for _, m in test_pairs])
    f_train = extract(train_imgs)                                        # 1. 良品特征
    n, gh, gw, d = f_train.shape
    bank_r = f_train.reshape(-1, d)                                      # 2. 记忆库
    core = mod.greedy_coreset(bank_r, max(1, int(len(bank_r) * keep)))   # 3. 核心集
    f_test = extract([im for im, _ in test_pairs])
    smap = mod.knn_scores(bank_r[core], f_test.reshape(-1, d), k=1).reshape(len(test_pairs), gh, gw)   # 4. 最近邻打分
    score = smap[:, 1:-1, 1:-1].reshape(len(smap), -1).max(axis=1)       # 5. 整图分数：去掉最外一圈再取最大（对照 1）
    heat = F.interpolate(torch.tensor(smap)[:, None], size=(64, 64), mode="bilinear", align_corners=False)[:, 0].numpy()   # 6. 上采样成热图
    print(f"设备 {device}｜patch 特征 {gh}x{gw}，{d} 维｜记忆库 {len(bank_r)} → 核心集 {len(core)}")
    print("Image AUROC（ResNet-18 layer2 + layer3）：", round(mod.auroc(score, y), 4))
    thr, esc, ovk = mod.threshold_at_escape(score[y == 0], score[y == 1], 0.05)
    print(f"漏检上限 5%：阈值 {thr:.3f}，实际漏检 {esc:.0%}，过检 {ovk:.0%}")
    fig, ax = plt.subplots(2, 4, figsize=(12, 6))
    for i in range(4):
        ax[0, i].imshow(test_pairs[i][0], cmap="gray"); ax[0, i].contour(test_pairs[i][1], colors="c"); ax[0, i].axis("off")
        ax[1, i].imshow(heat[i], cmap="magma", vmin=heat[:4].min(), vmax=heat[:4].max()); ax[1, i].set_title(f"分数 {score[i]:.2f}"); ax[1, i].axis("off")
    plt.show()
    return score

if img_scores is not None:
    try:
        resnet_scores = patchcore_resnet(train, test)
        if resnet_scores is not None:
            print("和前面 16 维像素块的结果比一比。缺陷只有两三个像素时，layer2 的一个位置对应 8x8 的输入像素，缺陷被稀释，")
            print("预训练特征在这种合成条纹上不一定更好（讲义第 3.7 节）；它的价值要在纹理复杂的真实图像上才显出来。")
    except NotImplementedError as exc:
        print("先完成作业里剩下的函数：", exc)

## 破坏实验与记录

- 训练集里混入 2 张带缺陷的图，AUROC 怎么变？为什么？
- 去掉局部聚合（`patch_features(im, agg=None)`，直接用 16 维 patch），结果怎么变？
- 把漏检约束下的过检率写进 `实验记录.md`。